In [ ]:
import torch
x = torch.rand(4,3)
x = torch.zeros(4,3,dtype=torch.float)
x = torch.ones(4,3,dtype=torch.float)
x = torch.tensor([1,2,3])

x = torch.rand(4,3,dtype=torch.float)
y = torch.rand_like(x)

x,y,x+y,torch.add(x,y)

(tensor([[0.6965, 0.4138, 0.9145],
         [0.7661, 0.3457, 0.5360],
         [0.3230, 0.9760, 0.2036],
         [0.0686, 0.6536, 0.3751]]),
 tensor([[0.6964, 0.5391, 0.7690],
         [0.8628, 0.3314, 0.9827],
         [0.1961, 0.9428, 0.3892],
         [0.8217, 0.9587, 0.1697]]),
 tensor([[1.3929, 0.9529, 1.6835],
         [1.6289, 0.6772, 1.5187],
         [0.5192, 1.9188, 0.5928],
         [0.8903, 1.6123, 0.5448]]))

In [23]:
import torch
# 维度变换，view，共享内存
x = torch.rand(3,4)
print(x)
y = x.view(12)
z = x.view(-1,6)
z[0,0] = 0

x,y,z

x = torch.rand(4,4)
y = x.clone().view(16)
y[0] = 0
x,y

tensor([[0.7474, 0.0908, 0.2063, 0.8870],
        [0.3949, 0.6912, 0.0690, 0.7759],
        [0.9377, 0.0965, 0.8443, 0.1989]])


(tensor([[0.9268, 0.6337, 0.1919, 0.5761],
         [0.0306, 0.2441, 0.4311, 0.3731],
         [0.1569, 0.3376, 0.7566, 0.6214],
         [0.2967, 0.2624, 0.2552, 0.2998]]),
 tensor([0.0000, 0.6337, 0.1919, 0.5761, 0.0306, 0.2441, 0.4311, 0.3731, 0.1569,
         0.3376, 0.7566, 0.6214, 0.2967, 0.2624, 0.2552, 0.2998]))

自动求导
pytorch会根据计算过程实时构建计算图
设x = 1
公式 z = (x+2) ** 2 + (x+1) + 3

首先在过程中，pytorch会把计算过程拆解成计算图，可以通过.grad_fn来查看，比如此时grad_fn为AddBackward0，求和操作，同时存在指向上一层的指针fn.next_functions，可以通过递归的方法来求出构造出来的计算图，然后通过.backward()来进行反向传播，计算dz/dx的梯度值，若z为矩阵，则计算雅可比矩阵，若z为值，则为每个xi计算梯度dz/dxi

doutput/dx = doutput/dz * dz/dyi * dyi/dxi = 1/16 * (2yi+1) * 1

In [58]:
import torch
# z = (x+2)**2 +3
x = torch.ones(4,4,requires_grad=True)
y = x + 2
z = y**2 + (y+3)

print(y.grad_fn,z.grad_fn)
def trace_graph(fn, depth=0):
    if fn is None: return
    print("  " * depth + str(fn))
    for next_fn, _ in fn.next_functions:
        trace_graph(next_fn, depth + 1)

print(trace_graph(z.grad_fn), z.grad_fn)

output = z.mean()
output.backward()
print(x.grad)

output3 = x.sum()
output3.backward()
print(x.grad)

<AddBackward0 object at 0x0000019BC3454820> <AddBackward0 object at 0x0000019BC2CD6D70>
None <AddBackward0 object at 0x0000019BC2CD6D70>
tensor([[0.4375, 0.4375, 0.4375, 0.4375],
        [0.4375, 0.4375, 0.4375, 0.4375],
        [0.4375, 0.4375, 0.4375, 0.4375],
        [0.4375, 0.4375, 0.4375, 0.4375]])
tensor([[1.4375, 1.4375, 1.4375, 1.4375],
        [1.4375, 1.4375, 1.4375, 1.4375],
        [1.4375, 1.4375, 1.4375, 1.4375],
        [1.4375, 1.4375, 1.4375, 1.4375]])
